# Borzoi Regulatory Sequence Prediction

Borzoi is a long-context regulatory sequence model published by Linder et al. (2024) that predicts regulatory signals from 524,288 bp DNA input windows. It extends the Enformer architecture with larger context, higher resolution output (6,144 bins), and replicate-specific checkpoints trained on different data splits. The tool provides two functions: single-replicate prediction for fast inference with one checkpoint, and ensemble prediction that runs all four replicates for robust consensus analysis with uncertainty estimates.

This notebook demonstrates both workflows, showing how to extract selected output tracks and compute cross-replicate statistics.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("borzoi")
display_overview("borzoi")
display_docs_section("borzoi", "Background")

# Borzoi

> Borzoi is a deep learning model that predicts gene expression and regulatory activity from a 524,288 bp DNA sequence. As the successor to Enformer, Borzoi uses dilated residual convolutional blocks combined with attention to achieve a 2.7x longer context window (524 kb vs. 196 kb), higher output resolution (6,144 bins), and improved accuracy, particularly for RNA-seq coverage prediction.

## Background

Gene regulation involves interactions across a wide range of genomic distances. While most promoter-proximal elements act within a few kilobases, enhancers can regulate genes from distances exceeding 100 kb, and [topologically associating domains](https://en.wikipedia.org/wiki/Topologically_associating_domain) (TADs) organize chromatin contacts across megabase scales. Capturing these long-range interactions requires models with sufficient context windows.

Borzoi was trained to predict RNA-seq coverage directly from DNA sequence, rather than the processed experimental tracks used by Enformer. This training objective enables:

* **Quantitative RNA-seq profiles**: Direct prediction of read coverage across gene bodies, capturing splicing patterns, alternative [TSS](https://en.wikipedia.org/wiki/Transcription_start_site) usage, and transcript isoform ratios
* **Broader regulatory context**: The 524 kb window captures most enhancer-promoter interactions and some TAD-level organization
* **Higher output resolution**: 6,144 bins at 128 bp resolution (vs. 896 bins in Enformer) provide finer spatial detail

The model also predicts [CAGE](https://en.wikipedia.org/wiki/Cap_analysis_of_gene_expression), [DNase-seq](https://en.wikipedia.org/wiki/DNase-Seq), [ATAC-seq](https://en.wikipedia.org/wiki/ATAC-seq), and [histone modification](https://en.wikipedia.org/wiki/Histone_modification) tracks, making it a general-purpose regulatory genomics predictor with improved long-range accuracy.

## Available tools

In [2]:
display_available_tools("borzoi")

- **`run_borzoi_ensemble()`** — Regulatory activity prediction using all 4 Borzoi replicates
- **`run_borzoi()`** — Regulatory activity prediction using a single Borzoi replicate

### `run_borzoi`

This function runs a single Borzoi replicate checkpoint on a 524,288 bp DNA sequence and returns a prediction matrix of shape [num_tracks, 6144]. It is the fastest option for regulatory activity prediction when cross-replicate uncertainty estimates are not required. The replicate index (0 through 3) and species (human or mouse) can be configured independently, and output tracks can be averaged into a single signal for compact downstream analysis.

In [3]:
import numpy as np
from proto_tools import (
    BORZOI_CONTEXT,
    BorzoiConfig,
    BorzoiInput,
    run_borzoi,
)

In [4]:
# Display input docs
display_api_reference("borzoi", "input", "run_borzoi")

# Create a full-length DNA sequence by repeating a short pattern to fill the required context
sequence = "ATCG" * (BORZOI_CONTEXT // 4)
inputs = BorzoiInput(sequences=[sequence])

**Input** — `BorzoiInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence` | `string` | required | DNA sequence for Borzoi inference. Must be exactly 524,288 bp and only contain valid nucleotide characters. |

In [5]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi")

single_config = BorzoiConfig(
    output_tracks=[0, 1, 2],
    species="human",
    replicate="0",
    avg_output_tracks=False,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `BorzoiConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `output_tracks` | `List[integer]` | `[0]` | Track indices to extract from model output. |
| `species` | `enum` | `human` | Species model to use. Available options: `human`, `mouse` |
| `replicate` | `enum` | `0` | Replicate ID to run. Available options: `0`, `1`, `2`, `3` |
| `avg_output_tracks` | `boolean` | `True` | Whether to average selected tracks. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device used for inference (inherited). |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [6]:
# Run the single-replicate prediction function
single_result = run_borzoi(inputs, single_config)

Running run_borzoi [00:00]

In [7]:
display_api_reference("borzoi", "output", "run_borzoi")

print(f"Tracks: {len(single_result.results[0].prediction)}")
print(f"Bins per track: {len(single_result.results[0].prediction[0])}")

**Output** — `BorzoiOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence` | `string` | required | Input DNA sequence that was scored. |
| `sequence_length` | `integer` | required | Length of the input sequence (always 524,288). |
| `prediction` | `List[array]` | required | Prediction matrix with shape `[num_tracks, 6144]`. |
| `output_tracks` | `List[integer]` | required | Track indices used for prediction. |
| `species` | `string` | required | Species used for prediction (`"human"` or `"mouse"`). |
| `replicate` | `string` | required | Borzoi replicate used (`"0"` through `"3"`). |
| `avg_output_tracks` | `boolean` | required | Whether requested tracks were averaged. |

Tracks: 3
Bins per track: 6144


### `run_borzoi_ensemble`

This function runs all four Borzoi replicate checkpoints and stacks the outputs into a tensor of shape [4, num_tracks, 6144]. Access to cross-replicate mean and standard deviation provides a measure of prediction confidence: regions where replicates agree strongly (low standard deviation) are more likely to reflect genuine regulatory signal than regions with high variance. When `avg_output_tracks` is enabled, each replicate's tracks are averaged before stacking, giving a final shape of [4, 1, 6144].

In [8]:
from proto_tools import (
    BorzoiEnsembleConfig,
    run_borzoi_ensemble,
)

In [9]:
# Display input docs (same BorzoiInput as single-replicate prediction)
display_api_reference("borzoi", "input", "run_borzoi_ensemble")

**Input** — `BorzoiInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence` | `string` | required | DNA sequence for Borzoi inference. Must be exactly 524,288 bp and only contain valid nucleotide characters. |

In [10]:
# Display config docs
display_api_reference("borzoi", "config", "run_borzoi_ensemble")

ensemble_config = BorzoiEnsembleConfig(
    output_tracks=[0, 1, 2],
    species="human",
    avg_output_tracks=True,
    device="cuda",  # Change to "cpu" if no GPU available
    verbose=False,
)

**Config** — `BorzoiEnsembleConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `output_tracks` | `List[integer]` | `[0]` | Track indices to extract from model output. |
| `species` | `enum` | `human` | Species model to use. Available options: `human`, `mouse` |
| `avg_output_tracks` | `boolean` | `True` | Whether to average selected tracks. |
| `seed` | `integer` |  | Random seed for reproducible results. When set, same seed + same input produces identical output. When None, a random seed is auto-assigned internally via the `resolved_seed` property. Some GPU tools produce approximately (not bit-exact) identical results because CUDA operations reduce floating-point values in non-deterministic thread order. Discrete outputs (sequences, structures) are exact; floating-point outputs (scores, coordinates) may differ at the bit level (\~1e-7). |
| `verbose` | `boolean` | `False` | Whether to print status messages. |
| `device` | `string` | `cuda` | Device used for inference. |
| `timeout` | `integer` | `600` | Maximum execution time in seconds. |

In [11]:
# Run the ensemble prediction function (runs all 4 replicates)
ensemble_result = run_borzoi_ensemble(inputs, ensemble_config)

Running run_borzoi [00:00]

Running run_borzoi [00:00]

Running run_borzoi [00:00]

Running run_borzoi [00:00]

In [12]:
display_api_reference("borzoi", "output", "run_borzoi_ensemble")

ensemble_array = np.array(ensemble_result.results[0].predictions)
mean_pred = ensemble_array.mean(axis=0)
std_pred = ensemble_array.std(axis=0)
print(f"Replicates: {ensemble_array.shape[0]}")
print(f"Mean shape: {mean_pred.shape}")
print(f"Std max: {std_pred.max():.4f}")

**Output** — `BorzoiEnsembleOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| `sequence` | `string` | required | Input DNA sequence that was scored. |
| `sequence_length` | `integer` | required | Length of the input sequence (always 524,288). |
| `predictions` | `List[array]` | required | Stacked predictions with shape `[4, num_tracks, 6144]` for replicates 0-3. |
| `output_tracks` | `List[integer]` | required | Track indices used for prediction. |
| `species` | `string` | required | Species used for prediction (`"human"` or `"mouse"`). |
| `avg_output_tracks` | `boolean` | required | Whether requested tracks were averaged. |
| `num_replicates` | `integer` |  | Number of replicates returned (always 4). |

Replicates: 4
Mean shape: (1, 6144)
Std max: 0.0051


## Export Results

Single-replicate and ensemble outputs can be saved to standard file formats for downstream analysis. Supported formats include JSON and CSV.

In [13]:
from pathlib import Path

output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

single_result.export(name="borzoi_single", export_path=output_dir, file_format="json")
print(f"Single-replicate result exported to {output_dir}/borzoi_single.json")

ensemble_result.export(name="borzoi_ensemble", export_path=output_dir, file_format="csv")
print(f"Ensemble summary exported to {output_dir}/borzoi_ensemble.csv")

Single-replicate result exported to example_output/borzoi_single.json
Ensemble summary exported to example_output/borzoi_ensemble.csv
